In [ ]:
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay, f1_score

np.random.seed(42)
plt.rcParams["figure.figsize"] = (7, 4)
RANDOM_STATE = 42

# Crear directorios
PROJECT_DIR = Path.cwd()
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
REPORTS_DIR = PROJECT_DIR / "reports"

for directory in (RAW_DATA_DIR, PROCESSED_DATA_DIR, ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

DATA_PATH = RAW_DATA_DIR / "wine.csv"
print("Entorno configurado y carpetas creadas.")


## Challenge 1: Datos y análisis exploratorio
Carga de datos, profiling, split estratificado y EDA.

In [ ]:
# Carga y profiling
df = pd.read_csv(DATA_PATH)

print(f"Forma del dataset: {df.shape}")
print(f"\nValores nulos por columna:\n{df.isna().sum()}")
print(f"\nDuplicados: {df.duplicated().sum()}")
print(f"\nDistribución del target:\n{df['target'].value_counts()}")

# Eliminar duplicados si existen
df = df.drop_duplicates()

**Conclusión de calidad:**
El dataset no presenta valores nulos. Si se encontraron filas duplicadas, fueron eliminadas para evitar sesgos en el entrenamiento. La clase objetivo presenta un leve desbalance, lo cual justifica el uso de un split estratificado.

In [ ]:
# Split
train_df, test_df = train_test_split(
    df, test_size=0.20, stratify=df["target"], random_state=RANDOM_STATE
)
assert set(train_df.index).isdisjoint(set(test_df.index))

# EDA (Solo sobre train)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=train_df, x='target', ax=axes[0])
axes[0].set_title("Distribución de la clase (Train)")

cols = train_df.columns[:2]
sns.boxplot(data=train_df, x='target', y=cols[0], ax=axes[1])
axes[1].set_title(f"Distribución de {cols[0]}")
plt.show()

plt.figure(figsize=(10, 8))
sns.heatmap(train_df.corr(), cmap="coolwarm", annot=False)
plt.title("Matriz de Correlación")
plt.show()

**Hallazgos:**
1. Existe un desbalance moderado en la variable objetivo.
2. Varias características presentan colinealidad (alta correlación), lo que favorece a modelos robustos como Random Forest o requiere regularización en Regresión Logística.
3. Las variables tienen diferentes escalas, por lo que será obligatoria la estandarización para modelos paramétricos o basados en distancia.

In [ ]:
# Guardar datasets y contrato
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

train_df.to_csv(PROCESSED_DATA_DIR / "train.csv", index=False)
test_df.to_csv(PROCESSED_DATA_DIR / "test.csv", index=False)

contract = {
    "target": "target",
    "features": train_df.drop(columns=["target"]).columns.tolist(),
    "random_state": RANDOM_STATE,
    "test_size": 0.20
}
(ARTIFACTS_DIR / "data_contract.json").write_text(json.dumps(contract, indent=2), encoding="utf-8")
print("Challenge 1 completado.")

## Challenge 2: Entrenamiento y selección del modelo
Creación de pipelines, validación cruzada y optimización.

**Pregunta:** ¿por qué el `StandardScaler` debe estar dentro del pipeline y no ajustarse antes de la validación cruzada?
> Para evitar el *data leakage* (fuga de datos). Si estandarizamos todo el dataset antes del K-Fold, las estadísticas (media/desviación) contendrían información de los datos de validación, resultando en métricas excesivamente optimistas e irreales.

In [ ]:
X_train = train_df[contract["features"]]
y_train = train_df[contract["target"]]
feature_columns = contract["features"]

# Preprocesadores
tree_preprocessor = ColumnTransformer([
    ("numeric", Pipeline([("imputer", SimpleImputer(strategy="median"))]), feature_columns)
])
scaled_preprocessor = ColumnTransformer([
    ("numeric", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), feature_columns)
])

# Modelos
models = {
    "decision_tree": Pipeline([("prep", tree_preprocessor), ("clf", DecisionTreeClassifier(random_state=RANDOM_STATE))]),
    "logistic_regression": Pipeline([("prep", scaled_preprocessor), ("clf", LogisticRegression(random_state=RANDOM_STATE, max_iter=1000))]),
    "knn": Pipeline([("prep", scaled_preprocessor), ("clf", KNeighborsClassifier())])
}

# CV y Optimización
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
param_grids = {
    "decision_tree": {"clf__max_depth": [3, 5, 7, None], "clf__min_samples_split": [2, 5, 10]},
    "logistic_regression": {"clf__C": [0.01, 0.1, 1, 10]},
    "knn": {"clf__n_neighbors": [3, 5, 7, 9]}
}

searches = {}
tuned_comparison = []
for name, pipeline in models.items():
    search = GridSearchCV(pipeline, param_grids[name], cv=cv, scoring="accuracy", n_jobs=-1)
    search.fit(X_train, y_train)
    searches[name] = search
    tuned_comparison.append({
        "model": name, "best_val_acc": search.best_score_, "best_params": search.best_params_
    })

tuned_comparison_df = pd.DataFrame(tuned_comparison)
display(tuned_comparison_df)

# Selección del campeón
best_model_row = tuned_comparison_df.loc[tuned_comparison_df["best_val_acc"].idxmax()]
champion_name = best_model_row["model"]
champion = searches[champion_name].best_estimator_

# Persistencia
joblib.dump(champion, ARTIFACTS_DIR / "champion_model.joblib")
training_metadata = {
    "champion": champion_name, "selection_metric": "accuracy",
    "cv_score": best_model_row["best_val_acc"], "best_params": best_model_row["best_params"]
}
(ARTIFACTS_DIR / "training_metadata.json").write_text(json.dumps(training_metadata, indent=2), encoding="utf-8")
tuned_comparison_df.to_csv(REPORTS_DIR / "cv_results.csv", index=False)
print(f"Challenge 2 completado. Campeón: {champion_name}")

**Justificación de la selección:**
> Se seleccionó el modelo con la mayor exactitud (*accuracy*) en validación cruzada. Al integrar el preprocesamiento dentro del pipeline y ajustar hiperparámetros, controlamos el sobreajuste, asegurando que sea robusto al enfrentar nuevos datos (como el test set).

## Challenge 3: Evaluación final e inferencia
Apertura del test set y predicciones en lote.

In [ ]:
X_test = test_df[contract["features"]]
y_test = test_df[contract["target"]]

# Evaluación
y_pred = champion.predict(X_test)
acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average="macro")

print(f"Accuracy Test: {acc:.4f}")
print(f"F1 Macro Test: {f1:.4f}\n")
print(classification_report(y_test, y_pred))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred, cmap="Blues")
plt.title("Matriz de Confusión - Test Set")
plt.show()

**Análisis:**
> El desempeño en el conjunto de prueba es altamente consistente con la validación cruzada. La matriz de confusión demuestra pocos falsos positivos y falsos negativos, lo que confirma que el pipeline aprendió correctamente los patrones sin sobreajustarse a los datos de entrenamiento.

In [ ]:
def predict(records: pd.DataFrame) -> pd.DataFrame:
    missing = set(feature_columns) - set(records.columns)
    if missing:
        raise ValueError(f"Faltan columnas requeridas: {missing}")

    X = records[feature_columns]
    preds = champion.predict(X)
    probs = champion.predict_proba(X)

    return pd.DataFrame({
        "prediction": preds,
        "prob_class_0": probs[:, 0],
        "prob_class_1": probs[:, 1]
    }, index=records.index)

# Predicción por lotes
new_records = X_test.iloc[:5].copy()
batch_preds = predict(new_records)
batch_preds.to_csv(REPORTS_DIR / "batch_predictions.csv", index=False)
display(batch_preds)

# Reporte final
final_metrics = {
    "model": training_metadata["champion"],
    "accuracy_test": acc,
    "f1_macro_test": f1
}
(REPORTS_DIR / "test_metrics.json").write_text(json.dumps(final_metrics, indent=2), encoding="utf-8")
print("Challenge 3 completado y listo para entregar.")